# Qwen3 MoE Node Pipeline With Imbalanced KV State

This notebook follows `qwen3_moe_node_pipeline.ipynb` and compares balanced and imbalanced `KVCacheState` inputs side by side.

## Experiment Scope

- Build a balanced `KVCacheState` with `build_kv_cache_state`
- Build an imbalanced `KVCacheState` with `build_imbalanced_kv_cache_state`
- Run normalize + codegen for both scenarios
- Run full hardware simulation for both scenarios and save traces under `trace/qwen3_moe_imbalanced_kv_state/`
- Run node-level simulation only for nodes that emit `FlashAttnOp`
- Compare KV cache fields, macro-op counts, flash-attention shapes, and latency deltas

`run_sim` is intentionally not used here because it rebuilds a balanced KV cache state internally.

In [1]:
import json
import logging
import re
from collections import Counter
from dataclasses import fields, is_dataclass
from math import ceil
from pathlib import Path
from pprint import pprint

import torch
from Desim import SimSession
from torch.fx import GraphModule

from nandmachine.commands.macro import FlashAttnOp, MacroOp
from nandmachine.config.config import NandConfig
from nandmachine.config.hardware_config import get_device_or_raise
from nandmachine.config.inference_config import InferenceConfig, MoEParallelConfig
from nandmachine.config.model_config import Qwen3MoEModelConfig
from nandmachine.frontend.core.graph.base import NxGraphMeta, NxTracer
from nandmachine.frontend.core.passes.cod_gen import CodeGenPass
from nandmachine.frontend.core.passes.normalize import NormalizePass
from nandmachine.frontend.network.qwen3_moe import Qwen3MoEDecoderLayer
from nandmachine.frontend.utlis import (
    build_imbalanced_kv_cache_state,
    build_kv_cache_state,
)
from nandmachine.simulator.entry_point import universe_run_sim
from nandmachine.simulator.hardware.xpu import xPU

MODEL_CARD_PATH = Path("model_cards/qwen3-moe-235B.json")
DEVICE_NAME = "H200_SXM"
COMPILE_MODE = "heuristic-GPU"
NUM_RANKS = 8
ATTN_DP_SIZE = 8
ATTN_TP_SIZE = 1
FFN_TP_SIZE = 1
FFN_EP_SIZE = 8
HBM_BANDWIDTH_GBPS = 3350
TRACE_DIR = Path("trace/qwen3_moe_imbalanced_kv_state")
SCENARIO_LABELS = ("balanced", "imbalanced")

assert MODEL_CARD_PATH.exists(), MODEL_CARD_PATH
assert NUM_RANKS > 0, NUM_RANKS
assert ATTN_DP_SIZE > 0, ATTN_DP_SIZE
assert ATTN_TP_SIZE > 0, ATTN_TP_SIZE
assert FFN_TP_SIZE > 0, FFN_TP_SIZE
assert FFN_EP_SIZE > 0, FFN_EP_SIZE


def kv_state_to_dict(kv_cache_state) -> dict[str, int | bool]:
    return {
        "total_kv_cache_size_per_layer": kv_cache_state.total_kv_cache_size_per_layer,
        "kv_block_size_tokens": kv_cache_state.kv_block_size_tokens,
        "num_kv_blocks": kv_cache_state.num_kv_blocks,
        "num_nand_pages_per_layer": kv_cache_state.num_nand_pages_per_layer,
        "num_hyper_pages_per_layer": kv_cache_state.num_hyper_pages_per_layer,
        "is_imbalance": kv_cache_state.is_imbalance,
    }


def macro_summary(macro_op_list: list[MacroOp]) -> Counter:
    return Counter(type(op).__name__ for op in macro_op_list)


def node_info(node: torch.fx.Node) -> dict[str, str]:
    return {
        "node_name": node.name,
        "node_op": node.op,
        "node_target": str(node.target),
    }


def _serialize_value(value: object) -> object:
    if isinstance(value, MacroOp):
        return {
            "id": value.id,
            "type": type(value).__name__,
        }
    if isinstance(value, list):
        return [_serialize_value(item) for item in value]
    if isinstance(value, tuple):
        return tuple(_serialize_value(item) for item in value)
    return value


def macro_info(op: MacroOp) -> dict[str, object]:
    assert is_dataclass(op), type(op)
    payload = {field.name: _serialize_value(getattr(op, field.name)) for field in fields(op)}
    payload["type"] = type(op).__name__
    return payload


def _sanitize_trace_name(raw_name: str) -> str:
    return re.sub(r"[^0-9A-Za-z._-]+", "_", raw_name).strip("._")


def trace_file_path(scenario_label: str, stem: str) -> Path:
    return TRACE_DIR / scenario_label / f"{_sanitize_trace_name(stem)}.json"


def run_macro_ops_with_trace(
    nand_config: NandConfig,
    commands: list[MacroOp],
    *,
    trace_path: Path,
    device_name: str,
    compile_mode: str,
    hbm_bandwidth_GBps: float,
) -> dict[str, object]:
    if not commands:
        raise ValueError("commands must not be empty")

    SimSession.reset()
    SimSession.init()

    hbm_bandwidth_bytes_per_sec = hbm_bandwidth_GBps * 10**9
    sim_xpu = xPU(
        nand_config,
        hbm_bandwidth_bytes_per_sec=hbm_bandwidth_bytes_per_sec,
        device_name=device_name,
        compile_mode=compile_mode,
        enable_trace=True,
    )
    sim_xpu.load_command(commands)
    SimSession.scheduler.run()

    final_time_ns = int(SimSession.sim_time.cycle)
    device = get_device_or_raise(device_name)
    final_cycle = ceil(final_time_ns * device.compute_module.clock_freq / 1e9)

    trace_path.parent.mkdir(parents=True, exist_ok=True)
    saved_trace_path = Path(sim_xpu.save_trace_file(str(trace_path)))

    tracer = sim_xpu.tracer
    assert tracer is not None
    complete_events = [event for event in tracer._events if event["ph"] == "X"]

    return {
        "cycle": final_cycle,
        "time_ns": final_time_ns,
        "trace_path": str(saved_trace_path),
        "trace_event_count": len(tracer._events),
        "trace_complete_event_count": len(complete_events),
        "trace_complete_event_categories": dict(
            Counter(event["cat"] for event in complete_events)
        ),
    }


def build_graph_artifacts(
    graph_meta: NxGraphMeta,
    raw_model_config: object,
    parallel_config: MoEParallelConfig,
) -> tuple[GraphModule, list[MacroOp]]:
    MacroOp._global_id_counter = 0

    with torch.device("meta"):
        model = Qwen3MoEDecoderLayer(raw_model_config, parallel_config)
        graph = NxTracer().trace(model)
        graph_module = GraphModule(model, graph)

    NormalizePass().transform(graph_module)
    graph_module.graph.meta = {CodeGenPass.GRAPH_META_KEY: graph_meta}
    CodeGenPass().transform(graph_module)

    macro_op_list = graph_module.graph.meta[CodeGenPass.MACRO_OP_LIST_META_KEY]
    return graph_module, macro_op_list


def flash_attn_shape_histogram(macro_op_list: list[MacroOp]) -> dict[str, int]:
    histogram = Counter(
        str(op.qk_bmm_shape)
        for op in macro_op_list
        if isinstance(op, FlashAttnOp)
    )
    return dict(histogram)


In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

model_card = json.loads(MODEL_CARD_PATH.read_text())
model_card.setdefault("attention_type", "gqa")
raw_model_config = type("Qwen3MoeConfig", (), model_card)()
model_config = Qwen3MoEModelConfig.from_config(raw_model_config)
nand_config = NandConfig(
    num_channels=6 * 8,
    num_plane=96,
    num_block=16,
    num_pages=16,
    tRead=4000,
    tWrite=4000 * 10,
    tErase=4000 * 100,
    page_size=4,
    sram_threshold=1024 * 80,
    enable_strict=True
)
parallel_config = MoEParallelConfig(
    num_ranks=NUM_RANKS,
    attn_dp_size=ATTN_DP_SIZE,
    attn_tp_size=ATTN_TP_SIZE,
    ffn_tp_size=FFN_TP_SIZE,
    ffn_ep_size=FFN_EP_SIZE,
)
inference_config = InferenceConfig(
    batch_size=1024,
    input_sequence_length=8 * 1024,
    output_sequence_length=1 * 1024,
    weight_bits=16,
    activation_bits=16,
    kv_cache_bits=16,
    kv_block_size_bytes=1024 * 256,
    memory_backend="nand",
    parallel_config=parallel_config,
)
runtime_simulator_config = {
    "device_name": DEVICE_NAME,
    "compile_mode": COMPILE_MODE,
    "hbm_bandwidth_GBps": HBM_BANDWIDTH_GBPS,
}
IMBALANCED_KV_BLOCK_SIZE_SWEEP_BYTES = (64 * 1024, 256 * 1024)


def build_inference_config_with_kv_block_size(kv_block_size_bytes: int) -> InferenceConfig:
    if kv_block_size_bytes <= 0:
        raise ValueError(
            f"kv_block_size_bytes must be positive, got {kv_block_size_bytes}"
        )
    return InferenceConfig(
        batch_size=inference_config.batch_size,
        input_sequence_length=inference_config.input_sequence_length,
        output_sequence_length=inference_config.output_sequence_length,
        weight_bits=inference_config.weight_bits,
        activation_bits=inference_config.activation_bits,
        kv_cache_bits=inference_config.kv_cache_bits,
        kv_block_size_bytes=kv_block_size_bytes,
        memory_backend=inference_config.memory_backend,
        parallel_config=parallel_config,
    )

TRACE_DIR.mkdir(parents=True, exist_ok=True)

balanced_kv_cache_state = build_kv_cache_state(
    nand_config,
    model_config,
    inference_config,
)
imbalanced_kv_cache_state = build_imbalanced_kv_cache_state(
    nand_config,
    model_config,
    inference_config,
)

kv_state_comparison = {
    "balanced": kv_state_to_dict(balanced_kv_cache_state),
    "imbalanced": kv_state_to_dict(imbalanced_kv_cache_state),
    "delta": {
        key: kv_state_to_dict(imbalanced_kv_cache_state)[key] - kv_state_to_dict(balanced_kv_cache_state)[key]
        for key in (
            "total_kv_cache_size_per_layer",
            "kv_block_size_tokens",
            "num_kv_blocks",
            "num_nand_pages_per_layer",
            "num_hyper_pages_per_layer",
        )
    },
}

print("trace output dir:", TRACE_DIR)
print("parallel config:", parallel_config)
print("runtime simulator config:", runtime_simulator_config)
pprint(kv_state_comparison)


trace output dir: trace/qwen3_moe_imbalanced_kv_state
parallel config: MoEParallelConfig(num_ranks=8, attn_dp_size=8, attn_tp_size=1, ffn_tp_size=1, ffn_ep_size=8)
runtime simulator config: {'device_name': 'H200_SXM', 'compile_mode': 'heuristic-GPU', 'hbm_bandwidth_GBps': 3350}
{'balanced': {'is_imbalance': False,
              'kv_block_size_tokens': 128,
              'num_hyper_pages_per_layer': 1024,
              'num_kv_blocks': 73728,
              'num_nand_pages_per_layer': 4718592,
              'total_kv_cache_size_per_layer': 19327352832},
 'delta': {'kv_block_size_tokens': 0,
           'num_hyper_pages_per_layer': -868,
           'num_kv_blocks': -62496,
           'num_nand_pages_per_layer': -3999744,
           'total_kv_cache_size_per_layer': -16382951424},
 'imbalanced': {'is_imbalance': True,
                'kv_block_size_tokens': 128,
                'num_hyper_pages_per_layer': 156,
                'num_kv_blocks': 11232,
                'num_nand_pages_per_layer

In [3]:
def collect_attention_node_reports(
    scenario_label: str,
    graph_module: GraphModule,
    *,
    nand_config: NandConfig,
    runtime_simulator_config: dict[str, object],
) -> list[dict[str, object]]:
    reports: list[dict[str, object]] = []

    for node_index, node in enumerate(graph_module.graph.nodes):
        node_macro_ops = node.meta.get("marco_op_list", [])
        flash_attn_ops = [op for op in node_macro_ops if isinstance(op, FlashAttnOp)]
        if not flash_attn_ops:
            continue

        trace_path = trace_file_path(
            scenario_label,
            f"node_{node_index:03d}_{node.name}_{node.op}",
        )
        sim_result = run_macro_ops_with_trace(
            nand_config,
            node_macro_ops,
            trace_path=trace_path,
            **runtime_simulator_config,
        )
        reports.append(
            {
                **node_info(node),
                "macro_op_count": len(node_macro_ops),
                "macro_op_summary": dict(macro_summary(node_macro_ops)),
                "flash_attn_count": len(flash_attn_ops),
                "flash_attn_shape_histogram": flash_attn_shape_histogram(node_macro_ops),
                "flash_attn_ops": [macro_info(op) for op in flash_attn_ops],
                "simulation": sim_result,
            }
        )

    return reports


def run_scenario(
    scenario_label: str,
    scenario_inference_config: InferenceConfig,
    kv_cache_state,
) -> dict[str, object]:
    graph_meta = NxGraphMeta(
        nand_config=nand_config,
        model_config=model_config,
        inference_config=scenario_inference_config,
        kv_cache_state=kv_cache_state,
    )
    graph_module, macro_op_list = build_graph_artifacts(
        graph_meta,
        raw_model_config,
        parallel_config,
    )
    full_simulation = run_macro_ops_with_trace(
        nand_config,
        macro_op_list,
        trace_path=trace_file_path(scenario_label, "full_simulation"),
        **runtime_simulator_config,
    )
    hbm_bandwidth_bytes_per_sec = runtime_simulator_config["hbm_bandwidth_GBps"] * 10**9
    full_model_summary_result = universe_run_sim(
        nand_config,
        model_config,
        scenario_inference_config,
        macro_op_list,
        hbm_bandwidth_bytes_per_sec=hbm_bandwidth_bytes_per_sec,
        device_name=runtime_simulator_config["device_name"],
        compile_mode=runtime_simulator_config["compile_mode"],
        xpu_type="default",
        kv_cache_state=kv_cache_state,
    )
    model_summary = {
        "layer_latency_ns": full_model_summary_result.layer_latency_ns,
        "model_latency_ns": full_model_summary_result.model_latency_ns,
        "model_throughput": full_model_summary_result.model_throughput,
        "throughput_per_GPU": full_model_summary_result.throughput_per_GPU,
        "kv_cache_total_size_GB": full_model_summary_result.kv_cache_total_size_GB,
    }
    attention_node_reports = collect_attention_node_reports(
        scenario_label,
        graph_module,
        nand_config=nand_config,
        runtime_simulator_config=runtime_simulator_config,
    )

    node_reports = []
    for node in graph_module.graph.nodes:
        node_macro_ops = node.meta.get("marco_op_list", [])
        if not node_macro_ops:
            continue
        node_reports.append(
            {
                **node_info(node),
                "macro_op_count": len(node_macro_ops),
                "macro_op_summary": dict(macro_summary(node_macro_ops)),
            }
        )

    return {
        "inference_config": {
            "kv_block_size_bytes": scenario_inference_config.kv_block_size_bytes,
        },
        "kv_cache_state": kv_state_to_dict(kv_cache_state),
        "node_count": len(list(graph_module.graph.nodes)),
        "node_reports": node_reports,
        "macro_op_count": len(macro_op_list),
        "macro_op_summary": dict(macro_summary(macro_op_list)),
        "flash_attn_count": sum(1 for op in macro_op_list if isinstance(op, FlashAttnOp)),
        "flash_attn_shape_histogram": flash_attn_shape_histogram(macro_op_list),
        "full_simulation": full_simulation,
        "model_summary": model_summary,
        "attention_node_simulations": attention_node_reports,
    }


def run_imbalanced_block_size_sweep(
    kv_block_size_bytes_options: tuple[int, ...],
) -> dict[str, dict[str, object]]:
    if not kv_block_size_bytes_options:
        raise ValueError("kv_block_size_bytes_options must not be empty")

    sweep_results: dict[str, dict[str, object]] = {}
    for kv_block_size_bytes in kv_block_size_bytes_options:
        scenario_inference_config = build_inference_config_with_kv_block_size(
            kv_block_size_bytes
        )
        kv_cache_state = build_imbalanced_kv_cache_state(
            nand_config,
            model_config,
            scenario_inference_config,
        )
        scenario_key = f"{kv_block_size_bytes // 1024}KiB"
        sweep_results[scenario_key] = run_scenario(
            f"imbalanced_kv_block_{scenario_key}",
            scenario_inference_config,
            kv_cache_state,
        )

    return sweep_results


In [4]:
scenario_results = {
    "balanced": run_scenario("balanced", inference_config, balanced_kv_cache_state),
    "imbalanced": run_scenario("imbalanced", inference_config, imbalanced_kv_cache_state),
}

scenario_comparison = {
    "kv_state_comparison": kv_state_comparison,
    "macro_op_count_delta": (
        scenario_results["imbalanced"]["macro_op_count"]
        - scenario_results["balanced"]["macro_op_count"]
    ),
    "flash_attn_count_delta": (
        scenario_results["imbalanced"]["flash_attn_count"]
        - scenario_results["balanced"]["flash_attn_count"]
    ),
    "full_simulation_time_ns_delta": (
        scenario_results["imbalanced"]["full_simulation"]["time_ns"]
        - scenario_results["balanced"]["full_simulation"]["time_ns"]
    ),
    "full_simulation_cycle_delta": (
        scenario_results["imbalanced"]["full_simulation"]["cycle"]
        - scenario_results["balanced"]["full_simulation"]["cycle"]
    ),
}

pprint(scenario_comparison)


2026-04-10 20:16:06,751 - INFO - Processed node=input_layernorm generated_macro_ops=1
2026-04-10 20:16:06,752 - INFO - Processed node=self_attn_qkv_proj generated_macro_ops=3
2026-04-10 20:16:06,752 - INFO - Processed node=self_attn_q_norm generated_macro_ops=1
2026-04-10 20:16:06,752 - INFO - Processed node=self_attn_k_norm generated_macro_ops=1
2026-04-10 20:16:06,752 - INFO - Processed node=self_attn_rotary_emb generated_macro_ops=0
2026-04-10 20:16:06,754 - INFO - Processed node=self_attn_attn generated_macro_ops=384
2026-04-10 20:16:06,754 - INFO - Processed node=self_attn_o_proj generated_macro_ops=3
2026-04-10 20:16:06,754 - INFO - Processed node=post_attention_layernorm generated_macro_ops=1
2026-04-10 20:16:06,755 - INFO - Processed node=mlp generated_macro_ops=119


VectorOp(id=1, input_ops=[], vector_op_type='rms_norm', vector_shape=[128, 4096], weight_bits=16)
MatMulOp(id=3, input_ops=[SramPrefetch(id=2, input_ops=[], num_prefetch_pages=18432), VectorOp(id=1, input_ops=[], vector_op_type='rms_norm', vector_shape=[128, 4096], weight_bits=16)], dim=(128, 4096, 9216), weight_bits=16)
VectorOp(id=5, input_ops=[MatMulOp(id=3, input_ops=[SramPrefetch(id=2, input_ops=[], num_prefetch_pages=18432), VectorOp(id=1, input_ops=[], vector_op_type='rms_norm', vector_shape=[128, 4096], weight_bits=16)], dim=(128, 4096, 9216), weight_bits=16)], vector_op_type='rms_norm', vector_shape=[128, 128], weight_bits=16)
VectorOp(id=6, input_ops=[VectorOp(id=5, input_ops=[MatMulOp(id=3, input_ops=[SramPrefetch(id=2, input_ops=[], num_prefetch_pages=18432), VectorOp(id=1, input_ops=[], vector_op_type='rms_norm', vector_shape=[128, 4096], weight_bits=16)], dim=(128, 4096, 9216), weight_bits=16)], vector_op_type='rms_norm', vector_shape=[128, 128], weight_bits=16)], vector_

2026-04-10 20:16:50,473 - INFO - Processed node=input_layernorm generated_macro_ops=1
2026-04-10 20:16:50,473 - INFO - Processed node=self_attn_qkv_proj generated_macro_ops=3
2026-04-10 20:16:50,473 - INFO - Processed node=self_attn_q_norm generated_macro_ops=1
2026-04-10 20:16:50,474 - INFO - Processed node=self_attn_k_norm generated_macro_ops=1
2026-04-10 20:16:50,474 - INFO - Processed node=self_attn_rotary_emb generated_macro_ops=0
2026-04-10 20:16:50,476 - INFO - Processed node=self_attn_attn generated_macro_ops=468
2026-04-10 20:16:50,476 - INFO - Processed node=self_attn_o_proj generated_macro_ops=3
2026-04-10 20:16:50,477 - INFO - Processed node=post_attention_layernorm generated_macro_ops=1
2026-04-10 20:16:50,478 - INFO - Processed node=mlp generated_macro_ops=119


MatMulOp(id=405, input_ops=[SramPrefetch(id=404, input_ops=[], num_prefetch_pages=3072)], dim=(64, 1536, 4096), weight_bits=16)
MatMulOp(id=408, input_ops=[SramPrefetch(id=407, input_ops=[], num_prefetch_pages=6144), MatMulOp(id=405, input_ops=[SramPrefetch(id=404, input_ops=[], num_prefetch_pages=3072)], dim=(64, 1536, 4096), weight_bits=16)], dim=(64, 4096, 3072), weight_bits=16)
VectorOp(id=410, input_ops=[], vector_op_type='silu_mul', vector_shape=[64, 1536], weight_bits=16)
MatMulOp(id=412, input_ops=[SramPrefetch(id=411, input_ops=[], num_prefetch_pages=3072)], dim=(64, 1536, 4096), weight_bits=16)
MatMulOp(id=415, input_ops=[SramPrefetch(id=414, input_ops=[], num_prefetch_pages=6144), MatMulOp(id=412, input_ops=[SramPrefetch(id=411, input_ops=[], num_prefetch_pages=3072)], dim=(64, 1536, 4096), weight_bits=16)], dim=(64, 4096, 3072), weight_bits=16)
VectorOp(id=417, input_ops=[], vector_op_type='silu_mul', vector_shape=[64, 1536], weight_bits=16)
MatMulOp(id=419, input_ops=[Sram

In [5]:
for scenario_label in SCENARIO_LABELS:
    result = scenario_results[scenario_label]
    print("=" * 120)
    print("scenario:", scenario_label)
    pprint(
        {
            "inference_config": result["inference_config"],
            "kv_cache_state": result["kv_cache_state"],
            "macro_op_count": result["macro_op_count"],
            "macro_op_summary": result["macro_op_summary"],
            "flash_attn_count": result["flash_attn_count"],
            "flash_attn_shape_histogram": result["flash_attn_shape_histogram"],
            "full_simulation": result["full_simulation"],
            "model_summary": result["model_summary"],
        }
    )
    print("attention node simulations:")
    for report in result["attention_node_simulations"]:
        pprint(
            {
                "node_name": report["node_name"],
                "node_op": report["node_op"],
                "node_target": report["node_target"],
                "macro_op_count": report["macro_op_count"],
                "macro_op_summary": report["macro_op_summary"],
                "flash_attn_count": report["flash_attn_count"],
                "flash_attn_shape_histogram": report["flash_attn_shape_histogram"],
                "simulation": report["simulation"],
            }
        )


scenario: balanced
{'flash_attn_count': 128,
 'flash_attn_shape_histogram': {'(288, 16, 128, 128)': 128},
 'full_simulation': {'cycle': 1505754,
                     'time_ns': 822816,
                     'trace_complete_event_categories': {'compute': 185,
                                                         'prefetch': 162,
                                                         'transfer': 2},
                     'trace_complete_event_count': 349,
                     'trace_event_count': 353,
                     'trace_path': 'trace/qwen3_moe_imbalanced_kv_state/balanced/full_simulation.json'},
 'inference_config': {'kv_block_size_bytes': 262144},
 'kv_cache_state': {'is_imbalance': False,
                    'kv_block_size_tokens': 128,
                    'num_hyper_pages_per_layer': 1024,
                    'num_kv_blocks': 73728,
                    'num_nand_pages_per_layer': 4718592,
                    'total_kv_cache_size_per_layer': 19327352832},
 'macro_op_count':

In [6]:
imbalanced_block_size_results = run_imbalanced_block_size_sweep(
    IMBALANCED_KV_BLOCK_SIZE_SWEEP_BYTES
)
imbalanced_block_size_comparison = {
    "64KiB": {
        "kv_block_size_bytes": imbalanced_block_size_results["64KiB"]["inference_config"]["kv_block_size_bytes"],
        "kv_cache_state": imbalanced_block_size_results["64KiB"]["kv_cache_state"],
        "macro_op_count": imbalanced_block_size_results["64KiB"]["macro_op_count"],
        "flash_attn_count": imbalanced_block_size_results["64KiB"]["flash_attn_count"],
        "full_simulation": imbalanced_block_size_results["64KiB"]["full_simulation"],
    },
    "256KiB": {
        "kv_block_size_bytes": imbalanced_block_size_results["256KiB"]["inference_config"]["kv_block_size_bytes"],
        "kv_cache_state": imbalanced_block_size_results["256KiB"]["kv_cache_state"],
        "macro_op_count": imbalanced_block_size_results["256KiB"]["macro_op_count"],
        "flash_attn_count": imbalanced_block_size_results["256KiB"]["flash_attn_count"],
        "full_simulation": imbalanced_block_size_results["256KiB"]["full_simulation"],
    },
    "delta_64KiB_minus_256KiB": {
        "kv_block_size_tokens": (
            imbalanced_block_size_results["64KiB"]["kv_cache_state"]["kv_block_size_tokens"]
            - imbalanced_block_size_results["256KiB"]["kv_cache_state"]["kv_block_size_tokens"]
        ),
        "num_kv_blocks": (
            imbalanced_block_size_results["64KiB"]["kv_cache_state"]["num_kv_blocks"]
            - imbalanced_block_size_results["256KiB"]["kv_cache_state"]["num_kv_blocks"]
        ),
        "macro_op_count": (
            imbalanced_block_size_results["64KiB"]["macro_op_count"]
            - imbalanced_block_size_results["256KiB"]["macro_op_count"]
        ),
        "flash_attn_count": (
            imbalanced_block_size_results["64KiB"]["flash_attn_count"]
            - imbalanced_block_size_results["256KiB"]["flash_attn_count"]
        ),
        "full_simulation_time_ns": (
            imbalanced_block_size_results["64KiB"]["full_simulation"]["time_ns"]
            - imbalanced_block_size_results["256KiB"]["full_simulation"]["time_ns"]
        ),
        "full_simulation_cycle": (
            imbalanced_block_size_results["64KiB"]["full_simulation"]["cycle"]
            - imbalanced_block_size_results["256KiB"]["full_simulation"]["cycle"]
        ),
    },
}

pprint(imbalanced_block_size_comparison)


2026-04-10 20:16:51,518 - INFO - Processed node=input_layernorm generated_macro_ops=1
2026-04-10 20:16:51,519 - INFO - Processed node=self_attn_qkv_proj generated_macro_ops=3
2026-04-10 20:16:51,519 - INFO - Processed node=self_attn_q_norm generated_macro_ops=1
2026-04-10 20:16:51,519 - INFO - Processed node=self_attn_k_norm generated_macro_ops=1
2026-04-10 20:16:51,519 - INFO - Processed node=self_attn_rotary_emb generated_macro_ops=0
2026-04-10 20:16:51,522 - INFO - Processed node=self_attn_attn generated_macro_ops=486
2026-04-10 20:16:51,523 - INFO - Processed node=self_attn_o_proj generated_macro_ops=3
2026-04-10 20:16:51,523 - INFO - Processed node=post_attention_layernorm generated_macro_ops=1
2026-04-10 20:16:51,523 - INFO - Processed node=mlp generated_macro_ops=119


VectorOp(id=1, input_ops=[], vector_op_type='rms_norm', vector_shape=[128, 4096], weight_bits=16)
MatMulOp(id=3, input_ops=[SramPrefetch(id=2, input_ops=[], num_prefetch_pages=18432), VectorOp(id=1, input_ops=[], vector_op_type='rms_norm', vector_shape=[128, 4096], weight_bits=16)], dim=(128, 4096, 9216), weight_bits=16)
VectorOp(id=5, input_ops=[MatMulOp(id=3, input_ops=[SramPrefetch(id=2, input_ops=[], num_prefetch_pages=18432), VectorOp(id=1, input_ops=[], vector_op_type='rms_norm', vector_shape=[128, 4096], weight_bits=16)], dim=(128, 4096, 9216), weight_bits=16)], vector_op_type='rms_norm', vector_shape=[128, 128], weight_bits=16)
VectorOp(id=6, input_ops=[VectorOp(id=5, input_ops=[MatMulOp(id=3, input_ops=[SramPrefetch(id=2, input_ops=[], num_prefetch_pages=18432), VectorOp(id=1, input_ops=[], vector_op_type='rms_norm', vector_shape=[128, 4096], weight_bits=16)], dim=(128, 4096, 9216), weight_bits=16)], vector_op_type='rms_norm', vector_shape=[128, 128], weight_bits=16)], vector_

2026-04-10 20:17:05,043 - INFO - Processed node=input_layernorm generated_macro_ops=1
2026-04-10 20:17:05,043 - INFO - Processed node=self_attn_qkv_proj generated_macro_ops=3
2026-04-10 20:17:05,044 - INFO - Processed node=self_attn_q_norm generated_macro_ops=1
2026-04-10 20:17:05,044 - INFO - Processed node=self_attn_k_norm generated_macro_ops=1
2026-04-10 20:17:05,044 - INFO - Processed node=self_attn_rotary_emb generated_macro_ops=0
2026-04-10 20:17:05,047 - INFO - Processed node=self_attn_attn generated_macro_ops=468
2026-04-10 20:17:05,047 - INFO - Processed node=self_attn_o_proj generated_macro_ops=3
2026-04-10 20:17:05,047 - INFO - Processed node=post_attention_layernorm generated_macro_ops=1
2026-04-10 20:17:05,048 - INFO - Processed node=mlp generated_macro_ops=119


FlashAttnOp(id=452, input_ops=[SramPrefetch(id=451, input_ops=[], num_prefetch_pages=4608)], qk_bmm_shape=(1152, 16, 128, 32), sv_bmm_shape=(1152, 16, 32, 128), softmax_shape=(18432, 32), weight_bits=16)
FlashAttnOp(id=455, input_ops=[SramPrefetch(id=454, input_ops=[], num_prefetch_pages=4608)], qk_bmm_shape=(1152, 16, 128, 32), sv_bmm_shape=(1152, 16, 32, 128), softmax_shape=(18432, 32), weight_bits=16)
FlashAttnOp(id=458, input_ops=[SramPrefetch(id=457, input_ops=[], num_prefetch_pages=4608)], qk_bmm_shape=(1152, 16, 128, 32), sv_bmm_shape=(1152, 16, 32, 128), softmax_shape=(18432, 32), weight_bits=16)
FlashAttnOp(id=461, input_ops=[SramPrefetch(id=460, input_ops=[], num_prefetch_pages=4608)], qk_bmm_shape=(1152, 16, 128, 32), sv_bmm_shape=(1152, 16, 32, 128), softmax_shape=(18432, 32), weight_bits=16)
FlashAttnOp(id=464, input_ops=[SramPrefetch(id=463, input_ops=[], num_prefetch_pages=4608)], qk_bmm_shape=(1152, 16, 128, 32), sv_bmm_shape=(1152, 16, 32, 128), softmax_shape=(18432, 3

In [7]:
for scenario_label, result in imbalanced_block_size_results.items():
    print("=" * 120)
    print("imbalanced kv block size scenario:", scenario_label)
    pprint(
        {
            "inference_config": result["inference_config"],
            "kv_cache_state": result["kv_cache_state"],
            "macro_op_count": result["macro_op_count"],
            "macro_op_summary": result["macro_op_summary"],
            "flash_attn_count": result["flash_attn_count"],
            "flash_attn_shape_histogram": result["flash_attn_shape_histogram"],
            "full_simulation": result["full_simulation"],
            "model_summary": result["model_summary"],
        }
    )
    print("attention node simulations:")
    for report in result["attention_node_simulations"]:
        pprint(
            {
                "node_name": report["node_name"],
                "node_op": report["node_op"],
                "node_target": report["node_target"],
                "macro_op_count": report["macro_op_count"],
                "macro_op_summary": report["macro_op_summary"],
                "flash_attn_count": report["flash_attn_count"],
                "flash_attn_shape_histogram": report["flash_attn_shape_histogram"],
                "simulation": report["simulation"],
            }
        )


imbalanced kv block size scenario: 64KiB
{'flash_attn_count': 162,
 'flash_attn_shape_histogram': {'(1152, 16, 128, 32)': 162},
 'full_simulation': {'cycle': 1937568,
                     'time_ns': 1058780,
                     'trace_complete_event_categories': {'compute': 219,
                                                         'prefetch': 196,
                                                         'transfer': 2},
                     'trace_complete_event_count': 417,
                     'trace_event_count': 421,
                     'trace_path': 'trace/qwen3_moe_imbalanced_kv_state/imbalanced_kv_block_64KiB/full_simulation.json'},
 'inference_config': {'kv_block_size_bytes': 65536},
 'kv_cache_state': {'is_imbalance': True,
                    'kv_block_size_tokens': 32,
                    'num_hyper_pages_per_layer': 162,
                    'num_kv_blocks': 46656,
                    'num_nand_pages_per_layer': 746496,
                    'total_kv_cache_size_per_layer

In [8]:
assert balanced_kv_cache_state.is_imbalance is False
assert imbalanced_kv_cache_state.is_imbalance is True
assert balanced_kv_cache_state.num_kv_blocks != imbalanced_kv_cache_state.num_kv_blocks
assert scenario_results["balanced"]["flash_attn_count"] > 0
assert scenario_results["imbalanced"]["flash_attn_count"] > 0
assert scenario_results["balanced"]["macro_op_count"] != scenario_results["imbalanced"]["macro_op_count"]
assert scenario_results["balanced"]["attention_node_simulations"]
assert scenario_results["imbalanced"]["attention_node_simulations"]
assert scenario_results["balanced"]["model_summary"]["layer_latency_ns"] == scenario_results["balanced"]["full_simulation"]["time_ns"]
assert scenario_results["imbalanced"]["model_summary"]["layer_latency_ns"] == scenario_results["imbalanced"]["full_simulation"]["time_ns"]
assert tuple(sorted(imbalanced_block_size_results)) == ("256KiB", "64KiB")
assert imbalanced_block_size_results["64KiB"]["inference_config"]["kv_block_size_bytes"] == 64 * 1024
assert imbalanced_block_size_results["256KiB"]["inference_config"]["kv_block_size_bytes"] == 256 * 1024
assert (
    imbalanced_block_size_results["64KiB"]["kv_cache_state"]["kv_block_size_tokens"]
    != imbalanced_block_size_results["256KiB"]["kv_cache_state"]["kv_block_size_tokens"]
)
assert (
    imbalanced_block_size_results["64KiB"]["macro_op_count"]
    != imbalanced_block_size_results["256KiB"]["macro_op_count"]
)
assert imbalanced_block_size_results["64KiB"]["attention_node_simulations"]
assert imbalanced_block_size_results["256KiB"]["attention_node_simulations"]
assert imbalanced_block_size_results["64KiB"]["model_summary"]["layer_latency_ns"] == imbalanced_block_size_results["64KiB"]["full_simulation"]["time_ns"]
assert imbalanced_block_size_results["256KiB"]["model_summary"]["layer_latency_ns"] == imbalanced_block_size_results["256KiB"]["full_simulation"]["time_ns"]

{
    "kv_state_comparison": kv_state_comparison,
    "balanced": {
        "macro_op_count": scenario_results["balanced"]["macro_op_count"],
        "flash_attn_count": scenario_results["balanced"]["flash_attn_count"],
        "flash_attn_shape_histogram": scenario_results["balanced"]["flash_attn_shape_histogram"],
        "full_simulation": scenario_results["balanced"]["full_simulation"],
        "model_summary": scenario_results["balanced"]["model_summary"],
        "attention_node_simulations": scenario_results["balanced"]["attention_node_simulations"],
    },
    "imbalanced": {
        "macro_op_count": scenario_results["imbalanced"]["macro_op_count"],
        "flash_attn_count": scenario_results["imbalanced"]["flash_attn_count"],
        "flash_attn_shape_histogram": scenario_results["imbalanced"]["flash_attn_shape_histogram"],
        "full_simulation": scenario_results["imbalanced"]["full_simulation"],
        "model_summary": scenario_results["imbalanced"]["model_summary"],
        "attention_node_simulations": scenario_results["imbalanced"]["attention_node_simulations"],
    },
    "delta": scenario_comparison,
    "imbalanced_kv_block_size_sweep": imbalanced_block_size_comparison,
}


{'kv_state_comparison': {'balanced': {'total_kv_cache_size_per_layer': 19327352832,
   'kv_block_size_tokens': 128,
   'num_kv_blocks': 73728,
   'num_nand_pages_per_layer': 4718592,
   'num_hyper_pages_per_layer': 1024,
   'is_imbalance': False},
  'imbalanced': {'total_kv_cache_size_per_layer': 2944401408,
   'kv_block_size_tokens': 128,
   'num_kv_blocks': 11232,
   'num_nand_pages_per_layer': 718848,
   'num_hyper_pages_per_layer': 156,
   'is_imbalance': True},
  'delta': {'total_kv_cache_size_per_layer': -16382951424,
   'kv_block_size_tokens': 0,
   'num_kv_blocks': -62496,
   'num_nand_pages_per_layer': -3999744,
   'num_hyper_pages_per_layer': -868}},
 'balanced': {'macro_op_count': 513,
  'flash_attn_count': 128,
  'flash_attn_shape_histogram': {'(288, 16, 128, 128)': 128},
  'full_simulation': {'cycle': 1505754,
   'time_ns': 822816,
   'trace_path': 'trace/qwen3_moe_imbalanced_kv_state/balanced/full_simulation.json',
   'trace_event_count': 353,
   'trace_complete_event_cou